In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys, os
_root = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(_root, "config.py")) and os.path.dirname(_root) != _root:
    _root = os.path.dirname(_root)
if _root not in sys.path: sys.path.insert(0, _root)

import config, pandas as pd

FIG_DIR = config.OUTPUTS_DIR / "exp01_continuous_9R" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

from cprics.io_data import load_peak_amplitudes
from cprics.continuous import build_all_combined
from cprics.tracking import track_islands
from cprics.plots import before_after_figure

pk  = load_peak_amplitudes(config.PEAK_AMP_DIR, "9_top_right")
isl = track_islands(build_all_combined(pk, wavelength=9))

# pick the site with the biggest early-warning lead (incumbent reaches 18 long after we do)
def lead(g):
    g = g.sort_values("date")
    hc = g.loc[g.hard_amplitude.ge(18), "date"]
    cc = g.loc[g.amplitude.ge(18), "date"]
    if g.hard_amplitude.max() < 30 or hc.empty or cc.empty:
        return -1
    return (hc.iloc[0] - cc.iloc[0]).days

site = isl.groupby("site_id").apply(lead).idxmax()

fig = before_after_figure(isl, site)
out = config.OUTPUTS_DIR / "figures"; out.mkdir(parents=True, exist_ok=True)
fig.write_html(FIG_DIR / "before_after.html", include_plotlyjs=True)
fig.show()
print("saved", FIG_DIR / "before_after.html", "| site:", site)

saved c:\Users\s2887048\Documents\network_rail\CPRICSv2\outputs\figures\before_after.html | site: CDM2_2100_9R_528


In [ ]:
from cprics.plots import early_warning_figure
fig, t = early_warning_figure(isl)
fig.write_html(FIG_DIR / "early_warning.html", include_plotlyjs=True)
fig.show()

print(f"median advance warning — Module 11: {t.module11.median():.1f} months, "
      f"C-PRICS: {t.cprics.median():.1f} months")
blind = t[t.module11 < 1]
print(f"sites Module 11 blindsided (<1 month warning): {len(blind)}, "
      f"of which C-PRICS warned >=3 months ahead: {(blind.cprics >= 3).sum()}")

median advance warning — Module 11: 0.0 months, C-PRICS: 0.0 months
sites Module 11 blindsided (<1 month warning): 21, of which C-PRICS warned >=3 months ahead: 3


In [5]:
import json, numpy as np
from cprics.features import build_feature_table
from cprics.classify import assign_archetype
CONCERN, CRITICAL = 18, 30

feats = build_feature_table(isl, min_obs=3).merge(
    isl.groupby("site_id")["amplitude"].max().rename("cont_max"), on="site_id")
with open(config.MODELS_DIR / "archetypes_9R.json") as fh:
    feats["archetype"] = assign_archetype(feats, json.load(fh))
arch = dict(zip(feats.site_id, feats.archetype))

rows = []
for sid, g in isl.groupby("site_id"):
    if sid not in arch:
        continue
    g = g.sort_values("date")
    critical = g.hard_amplitude.max() >= CRITICAL
    crit_date = g.loc[g.hard_amplitude >= CRITICAL, "date"].min() if critical else pd.NaT
    fd = g.loc[g.amplitude >= CONCERN, "date"]
    flagged = len(fd) > 0
    adv = flagged and (not critical or fd.min() < crit_date)
    lead = (crit_date - fd.min()).days / 30.44 if (critical and flagged and fd.min() < crit_date) else np.nan
    rows.append(dict(site_id=sid, archetype=arch[sid], critical=critical,
                     flagged=flagged, advance=adv, lead=lead))
E = pd.DataFrame(rows)

n, ncrit = len(E), int(E.critical.sum())
print(f"sites with history: {n} | critical: {ncrit} ({100*ncrit/n:.1f}%)\n")

TP = int((E.flagged & E.critical).sum());  FP = int((E.flagged & ~E.critical).sum())
FN = int((~E.flagged & E.critical).sum()); TN = int((~E.flagged & ~E.critical).sum())
print("Screening — flagged (continuous >=18 ever) vs became critical:")
print(f"  TP={TP}  FP={FP}  FN={FN}  TN={TN}")
print(f"  precision={TP/(TP+FP):.0%}   recall={TP/(TP+FN):.0%}\n")

aw = E[E.critical & E.advance]
print("Advance warning — flagged before going critical:")
print(f"  {len(aw)}/{ncrit} ({100*len(aw)/max(ncrit,1):.0f}%) of critical sites warned in advance")
if len(aw):
    print(f"  lead months: median {aw.lead.median():.1f} "
          f"(IQR {aw.lead.quantile(.25):.1f}–{aw.lead.quantile(.75):.1f})\n")

print("False positives (flagged, never critical) by archetype:")
print(E[E.flagged & ~E.critical].archetype.value_counts().to_string())

sites with history: 2263 | critical: 22 (1.0%)

Screening — flagged (continuous >=18 ever) vs became critical:
  TP=22  FP=69  FN=0  TN=2172
  precision=24%   recall=100%

Advance warning — flagged before going critical:
  12/22 (55%) of critical sites warned in advance
  lead months: median 12.0 (IQR 3.4–14.9)

False positives (flagged, never critical) by archetype:
archetype
Steady climber    34
Saw-tooth         34
Severe             1


In [ ]:
import plotly.graph_objects as go
cats = {
    "Became critical":                      int((E.flagged &  E.critical).sum()),
    "Saw-tooth — tamped before critical":   int((E.flagged & ~E.critical & (E.archetype == "Saw-tooth")).sum()),
    "Climber — rising, not yet critical":   int((E.flagged & ~E.critical & (E.archetype == "Steady climber")).sum()),
    "Severe — already high":                int((E.flagged & ~E.critical & (E.archetype == "Severe")).sum()),
}
cats = {k: v for k, v in cats.items() if v > 0}
colors = {"Became critical": "#c0392b", "Saw-tooth — tamped before critical": "#e67e22",
          "Climber — rising, not yet critical": "#f1c40f", "Severe — already high": "#8e44ad"}

fig = go.Figure(go.Bar(x=list(cats.values()), y=list(cats.keys()), orientation="h",
                       marker_color=[colors[k] for k in cats],
                       text=list(cats.values()), textposition="outside"))
fig.update_layout(
    title=f"The C-PRICS watchlist: {int(E.flagged.sum())} sites from {len(E)} on the network<br>"
          f"<sub>every site that became critical is on it — and the rest are real concerns, not noise</sub>",
    xaxis_title="number of sites", template="plotly_white", height=400, width=880,
    yaxis=dict(autorange="reversed"))
fig.write_html(FIG_DIR / "watchlist_composition.html", include_plotlyjs=True)
fig.show()

In [ ]:
import numpy as np, plotly.graph_objects as go

def advance_stats(amp_col):
    total, warned, leads = 0, 0, []
    for sid, g in isl.groupby("site_id"):
        g = g.sort_values("date")
        if g.hard_amplitude.max() < 30:
            continue
        total += 1
        crit = g.loc[g.hard_amplitude >= 30, "date"].iloc[0]
        d = g.loc[g[amp_col] >= 18, "date"]
        d = d[d < crit]
        if len(d):
            warned += 1
            leads.append((crit - d.iloc[0]).days / 30.44)
    return total, warned, leads

tot, m_w, m_l = advance_stats("hard_amplitude")   # Module 11 (flags D at hard >=18)
_,   c_w, c_l = advance_stats("amplitude")         # C-PRICS   (flags at continuous >=18)

fig = go.Figure()
fig.add_trace(go.Bar(x=["Module 11", "C-PRICS"], y=[m_w, c_w], name="warned in advance",
                     marker_color="#2e8b57", text=[m_w, c_w], textposition="auto"))
fig.add_trace(go.Bar(x=["Module 11", "C-PRICS"], y=[tot - m_w, tot - c_w],
                     name="no advance warning", marker_color="#c0392b",
                     text=[tot - m_w, tot - c_w], textposition="auto"))
fig.update_layout(barmode="stack",
    title=f"Advance warning on the {tot} sites that became critical<br>"
          f"<sub>median lead — Module 11: {np.median(m_l):.0f} months · C-PRICS: {np.median(c_l):.0f} months</sub>",
    yaxis_title="number of critical sites", template="plotly_white",
    width=640, height=480, legend=dict(x=1.02, y=1.0, xanchor="left", yanchor="top"))
fig.write_html(FIG_DIR / "advance_vs_module11.html", include_plotlyjs=True)
fig.show()
print(f"Module 11: {m_w}/{tot} in advance, median {np.median(m_l):.1f} mo")
print(f"C-PRICS:   {c_w}/{tot} in advance, median {np.median(c_l):.1f} mo")

Module 11: 11/31 in advance, median 8.3 mo
C-PRICS:   14/31 in advance, median 11.5 mo
